# Notebook de Jupyter para lógica de formulario de pago

Este notebook implementa y prueba la lógica de validación, cálculo y simulación de un formulario de pago similar al de tu proyecto web.

---

## 1. Configuración del entorno y librerías
Importar librerías necesarias y configurar opciones básicas.

In [ ]:
# 1. Configuración del entorno y librerías
import re
from datetime import datetime
from decimal import Decimal, ROUND_HALF_UP
from dataclasses import dataclass, field
from typing import List, Optional

# Opciones de impresión para decimales
Decimal128 = lambda x: Decimal(x).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)


## 2. Definición del modelo de datos del pago

Se definen las estructuras para cliente, tarjeta, dirección e ítems de compra.

In [ ]:
@dataclass
class Cliente:
    nombre: str
    email: str
    telefono: str

@dataclass
class Tarjeta:
    numero: str
    vencimiento: str  # MM/AA
    cvv: str
    titular: str

@dataclass
class ItemCompra:
    nombre: str
    cantidad: int
    precio_unitario: Decimal
    descuento: Decimal = Decimal('0.00')

@dataclass
class Pago:
    cliente: Cliente
    tarjeta: Optional[Tarjeta] = None
    metodo: str = 'tarjeta'  # 'tarjeta', 'banco', 'efectivo'
    banco: Optional[str] = None
    items: List[ItemCompra] = field(default_factory=list)
    subtotal: Decimal = Decimal('0.00')
    impuestos: Decimal = Decimal('0.00')
    total: Decimal = Decimal('0.00')
    fecha: Optional[datetime] = None
    transaccion: Optional[str] = None


## 3. Validación de campos del formulario

Funciones para validar nombre, email, número de tarjeta (Luhn), vencimiento, CVV y campos obligatorios.

In [ ]:
def validar_nombre(nombre: str) -> bool:
    return bool(nombre.strip())

def validar_email(email: str) -> bool:
    return re.match(r"^[\w\.-]+@[\w\.-]+\.\w+$", email) is not None

def luhn_checksum(card_number: str) -> bool:
    digits = [int(d) for d in card_number if d.isdigit()]
    checksum = 0
    parity = len(digits) % 2
    for i, digit in enumerate(digits):
        if i % 2 == parity:
            digit *= 2
            if digit > 9:
                digit -= 9
        checksum += digit
    return checksum % 10 == 0

def validar_tarjeta(numero: str) -> bool:
    return luhn_checksum(numero)

def validar_vencimiento(vencimiento: str) -> bool:
    try:
        mes, anio = vencimiento.split('/')
        mes = int(mes)
        anio = int('20' + anio) if len(anio) == 2 else int(anio)
        hoy = datetime.now()
        return 1 <= mes <= 12 and (anio > hoy.year or (anio == hoy.year and mes >= hoy.month))
    except Exception:
        return False

def validar_cvv(cvv: str) -> bool:
    return re.match(r"^\d{3,4}$", cvv) is not None


## 4. Cálculo de totales y generación de resumen

Función para calcular subtotal, impuestos y total, y para generar un resumen de compra similar al mostrado en la web.

In [ ]:
def calcular_totales(pago: Pago, impuesto_pct: Decimal = Decimal('0.19')) -> Pago:
    subtotal = sum((item.precio_unitario * item.cantidad - item.descuento) for item in pago.items)
    impuestos = (subtotal * impuesto_pct).quantize(Decimal('0.01'))
    total = (subtotal + impuestos).quantize(Decimal('0.01'))
    pago.subtotal = subtotal
    pago.impuestos = impuestos
    pago.total = total
    return pago

def generar_resumen(pago: Pago) -> str:
    resumen = f"""
Resumen de compra
-----------------
Cliente: {pago.cliente.nombre}
Email: {pago.cliente.email}
Método de pago: {pago.metodo}
Fecha: {pago.fecha.strftime('%d/%m/%Y %H:%M') if pago.fecha else '-'}

Productos:
"""
    for item in pago.items:
        resumen += f"- {item.nombre} x{item.cantidad}: ${item.precio_unitario} c/u"
        if item.descuento > 0:
            resumen += f" (desc: ${item.descuento})"
        resumen += "\n"
    resumen += f"\nSubtotal: ${pago.subtotal}\nImpuestos: ${pago.impuestos}\nTotal: ${pago.total}\n"
    if pago.metodo == 'efectivo':
        resumen += "\nDiríjase a un punto de pago con su número de factura para finalizar la compra.\n"
    return resumen

## 5. Pruebas de flujo de pago

Simulación de compras con tarjeta, banco y efectivo, mostrando validaciones y resumen generado.

In [ ]:
# Simulación de compra con tarjeta
cliente = Cliente(nombre="Macarena", email="maca@mail.com", telefono="123456789")
tarjeta = Tarjeta(numero="4539578763621486", vencimiento="12/29", cvv="123", titular="Macarena")
items = [
    ItemCompra(nombre="Camisa", cantidad=2, precio_unitario=Decimal128('12000')),
    ItemCompra(nombre="Pantalón", cantidad=1, precio_unitario=Decimal128('18000'), descuento=Decimal128('2000'))
]
pago = Pago(cliente=cliente, tarjeta=tarjeta, metodo='tarjeta', items=items, fecha=datetime.now())
pago = calcular_totales(pago)
print(generar_resumen(pago))

# Simulación de compra con banco
pago_banco = Pago(cliente=cliente, metodo='banco', banco='Banco Estado', items=items, fecha=datetime.now())
pago_banco = calcular_totales(pago_banco)
print(generar_resumen(pago_banco))

# Simulación de compra con efectivo
pago_efectivo = Pago(cliente=cliente, metodo='efectivo', items=items, fecha=datetime.now())
pago_efectivo = calcular_totales(pago_efectivo)
print(generar_resumen(pago_efectivo))